In [30]:
# =========================================================
# STAGE 3 — TEXT PREPROCESSING & FEATURE ENGINEERING
# Review Analytics and Suspicious Review Detection System
# =========================================================

# =========================================================
# 1. IMPORT LIBRARIES
# =========================================================

import pandas as pd
import numpy as np
import re
import string

from collections import Counter

# NLP
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Machine Learning
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

# Sparse Matrix Combination
from scipy.sparse import hstack
from scipy.sparse import csr_matrix

# Save models
import joblib

# Ignore warnings
import warnings
warnings.filterwarnings("ignore")

In [31]:
df = pd.read_csv("cleaned.csv")
display(df.head(5))
print("Dataset Shape:")
print(df.shape)
display(df.columns)
# print(df.head())

,Unnamed: 0,reviewerID,asin,reviewerName,reviewText,overall,summary,unixReviewTime,reviewTime,category,class,review_length,review_date
0,0,A2PAVURT4NOHE1,0000031852,Leah,Bought it for a ballet tutu but it is being wo...,5.0,Super cute,1388361600,"12 30, 2013",Sports_and_Outdoors,1.0,20,2013-12-30
1,1,A1SNLWGLFXD70K,0000031852,DEVA,I origonally didn't get the item I ordered. W...,4.0,Happy with purchase even though it came a lot ...,1392940800,"02 21, 2014",Sports_and_Outdoors,1.0,65,2014-02-21
2,2,A3URQ0LXLV46E9,0000031852,shortyvee,My daughter and her friends love the colors an...,4.0,zebralisous,1400544000,"05 20, 2014",Sports_and_Outdoors,1.0,22,2014-05-20
3,3,A1KJ4CVG87QW09,0000031852,Donna Carter-Scott,"Arrived very timely, cute grandbaby loves it. ...",4.0,Cute Tutu,1389657600,"01 14, 2014",Sports_and_Outdoors,1.0,25,2014-01-14
4,4,AA9ITO6ZLZW6,0000031852,Jazzy77,My little girl just loves to wear this tutu be...,5.0,Versatile,1399507200,"05 8, 2014",Sports_and_Outdoors,1.0,50,2014-05-08


Dataset Shape:
(199996, 13)


Index(['Unnamed: 0', 'reviewerID', 'asin', 'reviewerName', 'reviewText',
       'overall', 'summary', 'unixReviewTime', 'reviewTime', 'category',
       'class', 'review_length', 'review_date'],
      dtype='object')

In [3]:
# =========================================================
# 2. DOWNLOAD NLTK RESOURCES
# =========================================================

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package stopwords to C:\Users\Aron Varghese
[nltk_data]     John\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to C:\Users\Aron Varghese
[nltk_data]     John\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to C:\Users\Aron Varghese
[nltk_data]     John\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [4]:
# df = df.drop(columns=['Unnamed: 0','summary'])
print(df.columns)
display(df.isna().sum())

Index(['Unnamed: 0', 'reviewerID', 'asin', 'reviewerName', 'reviewText',
       'overall', 'summary', 'unixReviewTime', 'reviewTime', 'category',
       'class', 'review_length', 'review_date'],
      dtype='object')


Unnamed: 0          0
reviewerID          0
asin                0
reviewerName      811
reviewText          0
overall             0
summary             0
unixReviewTime      0
reviewTime          0
category            0
class               0
review_length       0
review_date         0
dtype: int64

In [32]:
# =========================================================
# 7. INITIALIZE NLP TOOLS
# =========================================================

stop_words = set(stopwords.words('english'))

lemmatizer = WordNetLemmatizer()

In [33]:
# =========================================================
# 8. TEXT CLEANING FUNCTION
# =========================================================

def clean_text(text):

    # -----------------------------------------------------
    # Convert to lowercase
    # -----------------------------------------------------

    text = text.lower()

    # -----------------------------------------------------
    # Remove URLs
    # -----------------------------------------------------

    text = re.sub(r'http\S+|www\S+', '', text)

    # -----------------------------------------------------
    # Remove HTML tags
    # -----------------------------------------------------

    text = re.sub(r'<.*?>', '', text)

    # -----------------------------------------------------
    # Remove punctuation
    # -----------------------------------------------------

    text = text.translate(
        str.maketrans('', '', string.punctuation)
    )

    # -----------------------------------------------------
    # Remove numbers
    # -----------------------------------------------------

    text = re.sub(r'\d+', '', text)

    # -----------------------------------------------------
    # Remove extra whitespace
    # -----------------------------------------------------

    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [34]:
# =========================================================
# 9. TOKENIZATION + STOPWORD REMOVAL + LEMMATIZATION
# =========================================================

def preprocess_text(text):

    # Clean basic text
    text = clean_text(text)

    # -----------------------------------------------------
    # Tokenization using regex
    # -----------------------------------------------------

    tokens = re.findall(r'\w+', text)

    # -----------------------------------------------------
    # Stopword Removal
    # -----------------------------------------------------

    tokens = [
        word
        for word in tokens
        if word not in stop_words
    ]

    # -----------------------------------------------------
    # Lemmatization
    # -----------------------------------------------------

    tokens = [
        lemmatizer.lemmatize(word)
        for word in tokens
    ]

    # -----------------------------------------------------
    # Join back to sentence
    # -----------------------------------------------------

    processed_text = " ".join(tokens)

    return processed_text


In [35]:
# =========================================================
# 10. CREATE PROCESSED REVIEW COLUMN
# =========================================================

print()
print("Processing Review Text...")

df['processed_review'] = df['reviewText'].apply(preprocess_text)
df['processed_review'].head(4)

print("Text Processing Completed")



Processing Review Text...
Text Processing Completed


In [14]:
# =========================================================
# 11. EXCLAMATION COUNT FEATURE
# =========================================================

def count_exclamations(text):

    return text.count('!')

df['exclamation_count'] = df['reviewText'].apply(
    count_exclamations
)

print()
print("Exclamation Count Feature Created")


Exclamation Count Feature Created


In [15]:
# =========================================================
# 12. CAPITAL LETTER RATIO FEATURE
# =========================================================

def capital_ratio(text):

    total_chars = len(text)

    if total_chars == 0:
        return 0

    capital_chars = sum(1 for c in text if c.isupper())

    return capital_chars / total_chars

df['capital_ratio'] = df['reviewText'].apply(
    capital_ratio
)

print()
print("Capital Ratio Feature Created")


Capital Ratio Feature Created


In [16]:
# =========================================================
# 13. REVIEWER REVIEW COUNT FEATURE
# =========================================================

reviewer_counts = df['reviewerID'].value_counts()

df['reviewer_review_count'] = df['reviewerID'].map(
    reviewer_counts
)

print()
print("Reviewer Activity Feature Created")


Reviewer Activity Feature Created


In [17]:
# =========================================================
# 14. PRODUCT REVIEW COUNT FEATURE
# =========================================================

product_counts = df['asin'].value_counts()

df['product_review_count'] = df['asin'].map(
    product_counts
)

print()
print("Product Review Count Feature Created")


Product Review Count Feature Created


In [18]:
# =========================================================
# 15. DUPLICATE REVIEW FEATURE
# =========================================================

df['is_duplicate_review'] = df.duplicated(
    subset=['processed_review'],
    keep=False
).astype(int)

print()
print("Duplicate Review Feature Created")


Duplicate Review Feature Created


In [19]:
# =========================================================
# 16. PROMOTIONAL WORD COUNT FEATURE
# =========================================================

promo_words = [
    'excellent',
    'perfect',
    'best',
    'amazing',
    'awesome',
    'great',
    'recommend',
    'highly',
    'fantastic',
    'love'
]

def promo_word_count(text):

    words = text.split()

    count = 0

    for word in words:

        if word in promo_words:
            count += 1

    return count

df['promo_word_count'] = df['processed_review'].apply(
    promo_word_count
)

print()
print("Promotional Word Count Feature Created")


Promotional Word Count Feature Created


In [20]:
# =========================================================
# 17. CHECK NEW FEATURES
# =========================================================

feature_columns = [
    'review_length',
    'exclamation_count',
    'capital_ratio',
    'reviewer_review_count',
    'product_review_count',
    'is_duplicate_review',
    'promo_word_count'
]

print()
print("Feature Preview:")

print(
    df[feature_columns].head()
)


Feature Preview:
   review_length  exclamation_count  capital_ratio  reviewer_review_count  \
0             20                  0       0.019048                      1   
1             65                  0       0.022857                      1   
2             22                  0       0.026316                      1   
3             25                  1       0.016000                      1   
4             50                  1       0.011952                      1   

   product_review_count  is_duplicate_review  promo_word_count  
0                     7                    0                 0  
1                     7                    0                 1  
2                     7                    0                 1  
3                     7                    0                 2  
4                     7                    0                 1  


In [22]:

# =========================================================
# 18. TF-IDF VECTORIZATION
# =========================================================

print()
print("Applying TF-IDF Vectorization...")

tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=2
)

X_text = tfidf.fit_transform(
    df['processed_review']
)

print("TF-IDF Shape:")
print(X_text.shape)


Applying TF-IDF Vectorization...
TF-IDF Shape:
(199996, 5000)
